# Traffic Sign Recognition for Kazakhstan
Five-class Computer Vision Assignment 2 project. Run on a Colab T4. Public mixed-source Baku dataset; no personal data collection claimed.

## 1 Download the public dataset
The collection includes GTSRB, RTSD, and additional Baku photographs.

In [1]:
# Download public mixed-source road sign images
import requests,zipfile,io,pathlib,time
ROOT=pathlib.Path('/content/traffic_sign_project');ROOT.mkdir(exist_ok=True)
r=requests.get('https://codeload.github.com/TN02/Road-Signs-Dataset/zip/refs/heads/main',timeout=(15,180));r.raise_for_status()
z=zipfile.ZipFile(io.BytesIO(r.content));z.extractall(ROOT)
DATA=ROOT/'Road-Signs-Dataset-main'
print('Downloaded images:',len(list((DATA/'JPEGImages').glob('*'))))


Downloaded images: 26978


## 2 Train and evaluate
HOG baseline, frozen DINOv2, augmentation ablation, group-based split, stress tests, nearest-example explanation.

In [2]:
import subprocess,sys
subprocess.check_call([sys.executable,'-m','pip','-q','install','scikit-image','scikit-learn','joblib','gradio==6.31.0','reportlab'])
"""Colab training and evaluation for five Kazakhstan-compatible traffic signs."""
import os, sys, json, time, random, pathlib, hashlib, re, shutil, platform
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
import torch
from torchvision import transforms
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from skimage.feature import hog
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(2)
ROOT=pathlib.Path('/content/traffic_sign_project')
DATA=ROOT/'Road-Signs-Dataset-main'
OUT=ROOT/'submission'; OUT.mkdir(exist_ok=True,parents=True)
FIG=OUT/'figures'; FIG.mkdir(exist_ok=True)
CLASSES=[
 {'source':'Main Road','code':'2.1','en':'Priority road','ru':'Главная дорога'},
 {'source':'Give Way','code':'2.4','en':'Give way','ru':'Уступите дорогу'},
 {'source':'STOP','code':'2.5','en':'Stop','ru':'Движение без остановки запрещено'},
 {'source':'No Entry','code':'3.1','en':'No entry','ru':'Въезд запрещен'},
 {'source':'Pedestrian Crossing','code':'5.16.1','en':'Pedestrian crossing','ru':'Пешеходный переход'}]
names=[c['en'] for c in CLASSES]
lookup={c['source'].lower():i for i,c in enumerate(CLASSES)}
(OUT/'classes.json').write_text(json.dumps(CLASSES,ensure_ascii=False,indent=2),encoding='utf-8')
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
hardware={'device':DEVICE,'gpu':torch.cuda.get_device_name(0) if DEVICE=='cuda' else None,
 'python':platform.python_version(),'torch':torch.__version__}
print('Hardware:',hardware,flush=True)

def dhash(im):
    a=np.asarray(im.convert('L').resize((9,8)))
    return int(''.join('1' if v else '0' for v in (a[:,1:]>a[:,:-1]).flatten()),2)

records=[]; raw=Counter(); bad=0
for ann in sorted((DATA/'Annotations').glob('*.xml')):
    try:
        x=ET.parse(ann).getroot(); fn=x.findtext('filename') or ann.stem+'.jpg'
        path=DATA/'JPEGImages'/pathlib.Path(fn).name
        objects=[o for o in x.findall('object') if o.findtext('name','').strip().lower() in lookup]
        if not objects: continue
        with Image.open(path) as inp:
            im=inp.convert('RGB'); w,h=im.size
            for o in objects:
                label=lookup[o.findtext('name').strip().lower()]; raw[label]+=1
                b=o.find('bndbox')
                x1,y1,x2,y2=[int(float(b.findtext(k))) for k in ['xmin','ymin','xmax','ymax']]
                x1,y1=max(0,x1),max(0,y1); x2,y2=min(w,x2),min(h,y2)
                if min(x2-x1,y2-y1)<16: continue
                crop=im.crop((x1,y1,x2,y2)).resize((128,128))
                base=re.split(r'\.rf\.',path.stem)[0]
                records.append({'label':label,'image':crop,'source':path.name,'base':base,'hash':dhash(crop)})
    except Exception: bad+=1
print('Raw selected boxes:',dict(raw),'usable:',len(records),'invalid files:',bad,flush=True)
assert all(any(r['label']==i for r in records) for i in range(5)), 'A selected class has no usable images'

# Group source filename variants and similar sign appearances before splitting.
# Tracks/physical-sign IDs are unavailable in this mixed collection.
selected=[]
for label in range(5):
    rows=[r for r in records if r['label']==label]
    random.shuffle(rows); rows=rows[:1200]
    parent=list(range(len(rows)))
    def find(i):
        while parent[i]!=i:
            parent[i]=parent[parent[i]]; i=parent[i]
        return i
    def union(i,j):
        a,b=find(i),find(j)
        if a!=b: parent[b]=a
    for i,r in enumerate(rows):
        for j in range(i):
            if r['base']==rows[j]['base'] or (r['hash']^rows[j]['hash']).bit_count()<=3:
                union(i,j)
    groups=defaultdict(list)
    for i,r in enumerate(rows): groups[find(i)].append(r)
    usable=list(groups.values()); random.shuffle(usable)
    assert len(usable)>=5, f'Too few independent appearance groups in {names[label]}'
    n=len(usable); nt=max(1,round(n*.15)); nv=max(1,round(n*.15))
    for gi,group in enumerate(usable):
        split='test' if gi<nt else 'val' if gi<nt+nv else 'train'
        # Cap each appearance group and each class/split to keep the run small.
        for r in group[:3]:
            r['group']=f'{label}_{gi}'; r['split']=split; selected.append(r)
cap={'train':300,'val':80,'test':100}
final=[]
for split in cap:
    for label in range(5):
        rows=[r for r in selected if r['split']==split and r['label']==label]
        random.shuffle(rows); final.extend(rows[:cap[split]])
records=final
for i,r in enumerate(records):
    rel=f"crops/{r['split']}/{r['label']}/{i:05d}.jpg"
    p=OUT/rel; p.parent.mkdir(parents=True,exist_ok=True); r['image'].save(p,quality=95)
    r['path']=rel
manifest=[{k:v for k,v in r.items() if k not in ('image','hash')} for r in records]
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2))
stats={s:[sum(r['split']==s and r['label']==i for r in records) for i in range(5)] for s in cap}
print('Split counts:',stats,flush=True)
train=[r for r in records if r['split']=='train']; val=[r for r in records if r['split']=='val']; test=[r for r in records if r['split']=='test']
assert set(r['group'] for r in train).isdisjoint(r['group'] for r in test)
yt=np.array([r['label'] for r in test])

fig,ax=plt.subplots(figsize=(8,3.5)); xx=np.arange(5)
for off,s in enumerate(['train','val','test']): ax.bar(xx+(off-1)*.25,stats[s],.25,label=s)
ax.set_xticks(xx,names,rotation=12); ax.set_ylabel('Crops'); ax.legend(); fig.tight_layout(); fig.savefig(FIG/'dataset_balance.png',dpi=160); plt.close(fig)
fig,axs=plt.subplots(2,5,figsize=(10,4))
for i in range(5):
    rows=[r for r in train if r['label']==i]
    for j in range(2): axs[j,i].imshow(rows[j%len(rows)]['image']); axs[j,i].axis('off'); axs[j,i].set_title(names[i],fontsize=9)
fig.tight_layout();fig.savefig(FIG/'examples.png',dpi=160);plt.close(fig)

def hog_feature(im):
    a=np.asarray(im.convert('L').resize((64,64)),dtype=np.float32)/255.
    return hog(a,orientations=9,pixels_per_cell=(8,8),cells_per_block=(2,2))
baseline=make_pipeline(StandardScaler(),LogisticRegression(max_iter=400,C=.1,random_state=SEED))
baseline.fit(np.stack([hog_feature(r['image']) for r in train]),[r['label'] for r in train])
pred_b=baseline.predict(np.stack([hog_feature(r['image']) for r in test]))
joblib.dump(baseline,OUT/'baseline.joblib')
print('Baseline macro-F1:',f1_score(yt,pred_b,average='macro'),flush=True)

print('Loading official DINOv2 ViT-S/14 pretrained encoder...',flush=True)
encoder=torch.hub.load('facebookresearch/dinov2','dinov2_vits14',pretrained=True).to(DEVICE).eval()
for p in encoder.parameters(): p.requires_grad=False
norm=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
augment=transforms.Compose([transforms.RandomPerspective(.15,p=.6),transforms.RandomRotation(12,fill=110),transforms.ColorJitter(.35,.3,.15,.03),transforms.GaussianBlur(3,sigma=(.1,1.0))])
@torch.inference_mode()
def features(images):
    out=[]
    for i in range(0,len(images),32):
        batch=torch.stack([norm(im) for im in images[i:i+32]]).to(DEVICE)
        out.append(encoder(batch).float().cpu())
    return torch.cat(out)
print('Extracting visual features...',flush=True)
X=features([r['image'] for r in train]); V=features([r['image'] for r in val]); T=features([r['image'] for r in test])
A=features([augment(r['image']) for r in train])
y=torch.tensor([r['label'] for r in train]); vy=torch.tensor([r['label'] for r in val])

def fit_head(x,y,tag):
    torch.manual_seed(SEED)
    mean=x.mean(0); std=x.std(0).clamp_min(.01)
    model=torch.nn.Linear(x.shape[1],5); opt=torch.optim.AdamW(model.parameters(),lr=.008,weight_decay=.01)
    curve=[]; best=None; bestloss=float('inf')
    for epoch in range(45):
        model.train(); opt.zero_grad(); loss=torch.nn.functional.cross_entropy(model((x-mean)/std),y); loss.backward();opt.step()
        model.eval()
        with torch.no_grad():
            vl=torch.nn.functional.cross_entropy(model((V-mean)/std),vy).item()
        curve.append([float(loss.item()),vl])
        if vl<bestloss: bestloss=vl;best={k:v.clone() for k,v in model.state_dict().items()}
    model.load_state_dict(best)
    with torch.no_grad(): pred=model((T-mean)/std).argmax(1).numpy()
    return model,mean,std,pred,curve
plain,pm,ps,pred_p,curve_p=fit_head(X,y,'plain')
model,mean,std,pred_m,curve=fit_head(torch.cat([X,A]),torch.cat([y,y]),'augmented')
torch.save({'state_dict':model.state_dict(),'mean':mean,'std':std,'classes':CLASSES},OUT/'dinov2_head.pt')
torch.save(encoder.state_dict(),OUT/'dinov2_vits14_backbone.pt')
torch.save({'features':X,'paths':[r['path'] for r in train],'labels':y},OUT/'retrieval.pt')

def metrics(pred): return {'accuracy':float(accuracy_score(yt,pred)),'macro_f1':float(f1_score(yt,pred,average='macro'))}
results={'baseline':metrics(pred_b),'dinov2_plain':metrics(pred_p),'dinov2_augmented':metrics(pred_m)}
robust={}
for cond in ['dark','blur']:
    ims=[ImageEnhance.Brightness(r['image']).enhance(.4) if cond=='dark' else r['image'].filter(ImageFilter.GaussianBlur(2)) for r in test]
    F=features(ims)
    with torch.no_grad():
        p0=plain((F-pm)/ps).argmax(1).numpy();p1=model((F-mean)/std).argmax(1).numpy()
    robust[cond]={'plain':metrics(p0),'augmented':metrics(p1)}
results['conditions']=robust
(OUT/'classification_report.json').write_text(json.dumps(classification_report(yt,pred_m,target_names=names,output_dict=True,zero_division=0),indent=2))
fig,ax=plt.subplots(figsize=(6,5)); ConfusionMatrixDisplay.from_predictions(yt,pred_m,display_labels=names,ax=ax,xticks_rotation=30,colorbar=False);fig.tight_layout();fig.savefig(FIG/'confusion_matrix.png',dpi=170);plt.close(fig)
fig,ax=plt.subplots(figsize=(7,3.5));ax.plot(np.array(curve)[:,0],label='Train');ax.plot(np.array(curve)[:,1],label='Validation');ax.set(xlabel='Epoch',ylabel='Cross entropy');ax.legend();fig.tight_layout();fig.savefig(FIG/'training_curves.png',dpi=170);plt.close(fig)

@torch.inference_mode()
def predict_image(im,condition='Normal'):
    im=im.convert('RGB')
    if condition=='Dark':im=ImageEnhance.Brightness(im).enhance(.4)
    if condition=='Blur':im=im.filter(ImageFilter.GaussianBlur(2))
    if DEVICE=='cuda':torch.cuda.synchronize()
    start=time.perf_counter(); f=features([im]); logits=model((f-mean)/std); probs=logits.double().softmax(1)[0].numpy()
    if DEVICE=='cuda':torch.cuda.synchronize()
    ms=(time.perf_counter()-start)*1000
    k=int(probs.argmax()); sim=torch.nn.functional.cosine_similarity(f,X); idx=sim.topk(min(3,len(train))).indices.tolist()
    return im,k,probs,ms,idx
lat=[]
predict_image(test[0]['image'])
for r in test[:20]:lat.append(predict_image(r['image'])[3])
results['latency_ms_mean']=float(np.mean(lat));results['latency_samples']=len(lat)
results['hardware']=hardware;results['counts']=stats;results['seed']=SEED;results['invalid_xml_or_images']=bad
(OUT/'metrics.json').write_text(json.dumps(results,indent=2))
print('RESULTS',json.dumps(results),flush=True)
fig,axs=plt.subplots(1,4,figsize=(10,3)); query,k,pr,ms,idx=predict_image(test[0]['image'])
axs[0].imshow(query);axs[0].set_title('Test query')
for a,j in zip(axs[1:],idx):a.imshow(train[j]['image']);a.set_title('Train: '+names[train[j]['label']],fontsize=9)
for a in axs:a.axis('off')
fig.tight_layout();fig.savefig(FIG/'explanation.png',dpi=170);plt.close(fig)

def build_app():
    import gradio as gr
    def infer(im,condition):
        if im is None:raise gr.Error('Upload or capture one cropped traffic sign.')
        shown,k,probs,ms,idx=predict_image(im,condition)
        c=CLASSES[k]
        rows=[[names[i],repr(float(probs[i]))] for i in np.argsort(-probs)]
        return f"{c['code']} {c['ru']}",rows,f'{ms:.1f} ms | {hardware["gpu"] or DEVICE} | Browser upload/snapshot',[(train[j]['image'],names[train[j]['label']]) for j in idx],shown
    with gr.Blocks(title='Traffic Sign Recognition for Kazakhstan') as demo:
        gr.Markdown('# Traffic Sign Recognition for Kazakhstan\nUpload a **cropped sign** or take a camera snapshot. Five supported sign types.\nPredictions use Kazakhstan sign codes and Russian sign names. This is an educational classifier, not a road safety system.')
        with gr.Row():
            with gr.Column():
                im=gr.Image(type='pil',sources=['upload','webcam'],label='One traffic sign')
                cond=gr.Radio(['Normal','Dark','Blur'],value='Normal',label='Simulate conditions')
                run=gr.Button('Recognize sign',variant='primary')
            with gr.Column():
                label=gr.Textbox(label='Sign identification')
                scores=gr.Dataframe(headers=['Class','Softmax score (0 to 1)'],datatype=['str','str'],type='array',interactive=False,label='Model scores - full stored precision')
                gr.Markdown('Scores are decimal model outputs, not calibrated confidence estimates.')
                latency=gr.Textbox(label='Latency and environment');shown=gr.Image(label='Processed input')
        gallery=gr.Gallery(label='Visual explanation - nearest training examples',columns=3)
        gr.Examples([[str(OUT/r['path']),'Normal'] for r in [next(t for t in test if t['label']==i) for i in range(5)]],inputs=[im,cond])
        gr.Markdown('Data: TN02/Road-Signs-Dataset, a mixture of public datasets and 230 Baku photographs. No claim of Kazakhstan field validation. Synthetic weather transformations are explicitly simulated. Unsupported objects can receive incorrect predictions.')
        if pathlib.Path('/content/Traffic_Sign_Project.zip').exists():
            gr.File(value='/content/Traffic_Sign_Project.zip',label='Download complete project')
            gr.File(value=str(OUT/'Report.pdf'),label='Download English report')
            if (OUT/'Demo.mp4').exists(): gr.File(value=str(OUT/'Demo.mp4'),label='Download real application demonstration')
        run.click(infer,[im,cond],[label,scores,latency,gallery,shown])
    return demo

ENGLISH_JS="""() => {
 const translations = {'Перетащите изображение сюда':'Drop an image here','Нажмите для загрузки':'Click to upload','- или -':'- or -','Загрузка':'Loading','Очистить':'Clear','Скачать':'Download','Редактировать':'Edit','Отмена':'Cancel','Сохранить':'Save','Настройки':'Settings','Создано с помощью Gradio':'Built with Gradio','Использовать через API':'Use via API'};
 const translate = () => {
   const walk = document.createTreeWalker(document.body, NodeFilter.SHOW_TEXT);
   let node; while (node = walk.nextNode()) { const key=node.nodeValue.trim(); if(translations[key]) node.nodeValue=node.nodeValue.replace(key,translations[key]); }
   document.querySelectorAll('[aria-label],[title]').forEach(el => { for(const attr of ['aria-label','title']) { const val=el.getAttribute(attr); if(translations[val]) el.setAttribute(attr,translations[val]); } });
 };
 translate(); new MutationObserver(translate).observe(document.body,{childList:true,subtree:true,characterData:true});
}"""

def force_english(demo):
    """Keep built-in Gradio labels English, independent of browser locale."""
    class EnglishLocale:
        def __init__(self, app): self.app=app
        async def __call__(self, scope, receive, send):
            if scope['type']=='http':
                scope=dict(scope)
                scope['headers']=[(k,v) for k,v in scope.get('headers',[]) if k.lower()!=b'accept-language']+[(b'accept-language',b'en')]
            await self.app(scope,receive,send)
    demo.app.middleware_stack=EnglishLocale(demo.app.middleware_stack)

print('TRAINING_COMPLETE',flush=True)


{
  "baseline": {
    "accuracy": 0.997867803837953,
    "macro_f1": 0.9975561259535086
  },
  "dinov2_plain": {
    "accuracy": 0.997867803837953,
    "macro_f1": 0.99799994999875
  },
  "dinov2_augmented": {
    "accuracy": 0.997867803837953,
    "macro_f1": 0.99799994999875
  },
  "conditions": {
    "dark": {
      "plain": {
        "accuracy": 0.9936034115138592,
        "macro_f1": 0.9935642789405689
      },
      "augmented": {
        "accuracy": 0.997867803837953,
        "macro_f1": 0.9975451211097794
      }
    },
    "blur": {
      "plain": {
        "accuracy": 0.9957356076759062,
        "macro_f1": 0.9960000000000001
      },
      "augmented": {
        "accuracy": 0.9957356076759062,
        "macro_f1": 0.9959998999974999
      }
    }
  },
  "latency_ms_mean": 10.094115900011502,
  "latency_samples": 20,
  "hardware": {
    "device": "cuda",
    "gpu": "Tesla T4",
    "python": "3.13.16",
    "torch": "2.14.1+cu130"
  },
  "counts": {
    "train": [
      300,
   

## 3 Create the English report and captioned demo
All quantitative content is read from the measured metrics.

In [ ]:
# Package architecture and saved-model app; build the report only
import pathlib,shutil,torch
cache=pathlib.Path(torch.hub.get_dir())/'facebookresearch_dinov2_main'
arch=OUT/'dinov2_source'
shutil.copytree(cache/'dinov2',arch/'dinov2',dirs_exist_ok=True)
for name in ['hubconf.py','LICENSE']:shutil.copy(cache/name,arch/name)
APP_SOURCE='"""Run from the extracted submission directory: python app.py."""\nimport pathlib,json,time,os\nimport numpy as np\nfrom PIL import ImageEnhance,ImageFilter,Image\nimport torch\nfrom torchvision import transforms\nimport gradio as gr\n\nROOT=pathlib.Path(__file__).resolve().parent\ndevice=os.environ.get(\'TRAFFIC_SIGN_DEVICE\',\'cuda\' if torch.cuda.is_available() else \'cpu\')\ntorch.set_num_threads(2)\narchitecture=ROOT/\'dinov2_source\'\nif not (architecture/\'hubconf.py\').exists():\n    raise FileNotFoundError(\'Missing bundled DINOv2 architecture. Extract the complete project ZIP before launching.\')\nenc=torch.hub.load(str(architecture),\'dinov2_vits14\',source=\'local\',pretrained=False).to(device).eval()\nenc.load_state_dict(torch.load(ROOT/\'dinov2_vits14_backbone.pt\',map_location=device,weights_only=True))\nck=torch.load(ROOT/\'dinov2_head.pt\',map_location=\'cpu\',weights_only=True)\nhead=torch.nn.Linear(384,5);head.load_state_dict(ck[\'state_dict\']);head.eval()\nmean,std=ck[\'mean\'],ck[\'std\'];classes=ck[\'classes\']\nretr=torch.load(ROOT/\'retrieval.pt\',map_location=\'cpu\',weights_only=True)\nnorm=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize([.485,.456,.406],[.229,.224,.225])])\n@torch.inference_mode()\ndef infer(im,condition):\n    if im is None:raise gr.Error(\'Upload or capture one cropped sign.\')\n    im=im.convert(\'RGB\')\n    if condition==\'Dark\':im=ImageEnhance.Brightness(im).enhance(.4)\n    if condition==\'Blur\':im=im.filter(ImageFilter.GaussianBlur(2))\n    if device==\'cuda\':torch.cuda.synchronize()\n    t=time.perf_counter();f=enc(norm(im).unsqueeze(0).to(device)).float().cpu()\n    probs=head((f-mean)/std).double().softmax(1)[0].numpy()\n    if device==\'cuda\':torch.cuda.synchronize()\n    ms=(time.perf_counter()-t)*1000;k=int(probs.argmax());c=classes[k]\n    ids=torch.nn.functional.cosine_similarity(f,retr[\'features\']).topk(3).indices.tolist()\n    examples=[(Image.open(ROOT/retr[\'paths\'][i]).copy(),classes[int(retr[\'labels\'][i])][\'en\']) for i in ids]\n    env=torch.cuda.get_device_name(0) if device==\'cuda\' else \'CPU\'\n    rows=[[classes[i][\'en\'],repr(float(probs[i]))] for i in np.argsort(-probs)]\n    return f"{c[\'code\']} {c[\'ru\']}",rows,f\'{ms:.1f} ms | {env}\',examples,im\nwith gr.Blocks(title=\'Traffic Sign Recognition for Kazakhstan\') as demo:\n    gr.Markdown(\'# Traffic Sign Recognition for Kazakhstan\\nUpload or capture **one cropped sign**. Five supported classes. Educational demonstration.\')\n    with gr.Row():\n        with gr.Column():\n            image=gr.Image(type=\'pil\',sources=[\'upload\',\'webcam\'],label=\'One traffic sign\')\n            condition=gr.Radio([\'Normal\',\'Dark\',\'Blur\'],value=\'Normal\',label=\'Simulate conditions\')\n            run=gr.Button(\'Recognize sign\',variant=\'primary\')\n        with gr.Column():\n            label=gr.Textbox(label=\'Sign identification\')\n            scores=gr.Dataframe(headers=[\'Class\',\'Softmax score (0 to 1)\'],datatype=[\'str\',\'str\'],type=\'array\',interactive=False,label=\'Model scores - full stored precision\')\n            gr.Markdown(\'Scores are displayed as decimal numbers without integer percentage rounding. They are not calibrated confidence estimates.\')\n            latency=gr.Textbox(label=\'Latency and environment\');shown=gr.Image(label=\'Processed input\')\n    gallery=gr.Gallery(label=\'Nearest training examples\',columns=3)\n    run.click(infer,[image,condition],[label,scores,latency,gallery,shown])\n    manifest=json.loads((ROOT/\'manifest.json\').read_text())\n    samples=[next(r for r in manifest if r[\'split\']==\'test\' and r[\'label\']==i) for i in range(5)]\n    gr.Examples([[str(ROOT/r[\'path\']),\'Normal\'] for r in samples],inputs=[image,condition])\n    gr.Markdown(\'Data: TN02/Road-Signs-Dataset; mixed public data and additional Baku photos. Kazakhstan field performance is unverified. Unsupported inputs may receive incorrect labels. Dark and Blur are simulated transformations.\')\ndef force_english(demo):\n    class EnglishLocale:\n        def __init__(self, app): self.app=app\n        async def __call__(self,scope,receive,send):\n            if scope[\'type\']==\'http\':\n                scope=dict(scope)\n                scope[\'headers\']=[(k,v) for k,v in scope.get(\'headers\',[]) if k.lower()!=b\'accept-language\']+[(b\'accept-language\',b\'en\')]\n            await self.app(scope,receive,send)\n    demo.app.middleware_stack=EnglishLocale(demo.app.middleware_stack)\nENGLISH_JS="""() => {\n const translations = {\'Перетащите изображение сюда\':\'Drop an image here\',\'Нажмите для загрузки\':\'Click to upload\',\'- или -\':\'- or -\',\'Загрузка\':\'Loading\',\'Очистить\':\'Clear\',\'Скачать\':\'Download\',\'Редактировать\':\'Edit\',\'Отмена\':\'Cancel\',\'Сохранить\':\'Save\',\'Настройки\':\'Settings\',\'Создано с помощью Gradio\':\'Built with Gradio\',\'Использовать через API\':\'Use via API\'};\n const translate=()=>{const walk=document.createTreeWalker(document.body,NodeFilter.SHOW_TEXT);let node;while(node=walk.nextNode()){const key=node.nodeValue.trim();if(translations[key])node.nodeValue=node.nodeValue.replace(key,translations[key]);}document.querySelectorAll(\'[aria-label],[title]\').forEach(el=>{for(const attr of [\'aria-label\',\'title\']){const val=el.getAttribute(attr);if(translations[val])el.setAttribute(attr,translations[val]);}});};\n translate();new MutationObserver(translate).observe(document.body,{childList:true,subtree:true,characterData:true});\n}"""\nif __name__==\'__main__\':\n    import threading\n    demo.launch(prevent_thread_lock=True,js=ENGLISH_JS,css=\'footer {display:none !important}\')\n    force_english(demo)\n    threading.Event().wait()\n'
REPORT_SOURCE='"""Generate the English report and demonstration from measured experiment outputs."""\nimport pathlib,json,sys,shutil,textwrap\nfrom reportlab.platypus import SimpleDocTemplate,Paragraph,Spacer,Table,TableStyle,Image as RLImage,PageBreak\nfrom reportlab.lib.styles import getSampleStyleSheet,ParagraphStyle\nfrom reportlab.lib import colors\nfrom reportlab.lib.enums import TA_LEFT\nfrom reportlab.graphics.shapes import Drawing,Rect,Line,String,Polygon\nfrom PIL import Image,ImageDraw,ImageFont\nimport numpy as np\n\nOUT=pathlib.Path(__file__).resolve().parent\nM=json.loads((OUT/\'metrics.json\').read_text())\nF=OUT/\'figures\'\nstyles=getSampleStyleSheet()\nstyles[\'Normal\'].fontName=\'Helvetica\';styles[\'Normal\'].fontSize=10.5;styles[\'Normal\'].leading=15\nstyles[\'Normal\'].spaceAfter=10\nstyles[\'Title\'].fontName=\'Helvetica-Bold\';styles[\'Title\'].fontSize=25;styles[\'Title\'].leading=30;styles[\'Title\'].textColor=colors.black\nfor n in [\'Heading1\',\'Heading2\']:styles[n].textColor=colors.black\nstyles.add(ParagraphStyle(name=\'SmallText\',parent=styles[\'Normal\'],fontSize=8.5,leading=12))\nstory=[]\ndef p(t,style=\'Normal\'):story.append(Paragraph(t,styles[style]))\ndef gap(h=8):story.append(Spacer(1,h))\ndef heading(t):p(t,\'Heading1\')\ndef page():story.append(PageBreak())\ndef figure(name,w=460):\n    im=Image.open(F/name); h=w*im.height/im.width\n    story.append(RLImage(str(F/name),width=w,height=h));gap()\ndef table(rows,widths=None):\n    rows=[[Paragraph(str(x),styles[\'SmallText\']) for x in row] for row in rows]\n    t=Table(rows,colWidths=widths,hAlign=\'LEFT\');t.setStyle(TableStyle([\n      (\'BACKGROUND\',(0,0),(-1,0),colors.HexColor(\'#e8edf2\')),(\'VALIGN\',(0,0),(-1,-1),\'TOP\'),\n      (\'LINEBELOW\',(0,0),(-1,0),.6,colors.grey),(\'BOTTOMPADDING\',(0,0),(-1,-1),7),(\'TOPPADDING\',(0,0),(-1,-1),7)]));story.append(t);gap()\ndef foot(canvas,doc):\n    canvas.setFont(\'Helvetica\',8);canvas.setFillColor(colors.grey);canvas.drawString(50,30,\'Computer Vision Assignment 2\');canvas.drawRightString(545,30,str(doc.page))\n\ndef method_diagram():\n    d=Drawing(470,180)\n    def box(x,y,w,h,title,subtitle=\'\'):\n        d.add(Rect(x,y,w,h,rx=4,fillColor=colors.HexColor(\'#edf2f7\'),strokeColor=colors.HexColor(\'#60748a\')))\n        d.add(String(x+w/2,y+h/2+(4 if subtitle else -3),title,textAnchor=\'middle\',fontName=\'Helvetica-Bold\',fontSize=8))\n        if subtitle:d.add(String(x+w/2,y+10,subtitle,textAnchor=\'middle\',fontSize=7))\n    def arrow(x1,y1,x2,y2):\n        d.add(Line(x1,y1,x2,y2,strokeColor=colors.HexColor(\'#334b63\')))\n        d.add(Polygon([x2,y2,x2-5,y2+3,x2-5,y2-3],fillColor=colors.HexColor(\'#334b63\'),strokeColor=None))\n    d.add(String(0,166,\'Published DINOv2 pretraining\',fontName=\'Helvetica-Bold\',fontSize=10))\n    box(0,106,105,40,\'Image views\',\'unlabeled images\')\n    box(145,128,115,30,\'Teacher encoder\',\'EMA weights\')\n    box(145,84,115,30,\'Student encoder\',\'learned weights\')\n    box(302,106,150,40,\'Self supervision\',\'global and patch objectives\')\n    arrow(105,126,145,143);arrow(105,126,145,99);arrow(260,143,302,126);arrow(260,99,302,126)\n    d.add(String(0,63,\'This project uses the pretrained encoder\',fontName=\'Helvetica-Bold\',fontSize=10))\n    for x,w,t,s in [(0,85,\'Sign crop\',\'224 x 224 RGB\'),(112,100,\'Frozen ViT S/14\',\'16 x 16 patches\'),(239,85,\'Embedding\',\'384 features\'),(351,101,\'Linear head\',\'5 class scores\')]:\n        box(x,9,w,39,t,s)\n    for a,b in [(85,112),(212,239),(324,351)]:arrow(a,28,b,28)\n    story.append(d);gap()\n\np(\'Traffic Sign Recognition for Kazakhstan\',\'Title\')\np(\'A five class image classifier using DINOv2 and a Baku road sign collection\',\'Heading2\')\np(\'Computer Vision Assignment 2 | Astana IT University | 7 November 2026\')\nheading(\'Abstract\')\np(\'This project compares a handcrafted HOG baseline with a frozen DINOv2 ViT-S/14 visual encoder and a trained linear classifier. Five sign types were selected from the TN02 Road Signs Dataset: priority road, give way, stop, no entry, and pedestrian crossing. The interface maps predictions to Kazakhstan sign identifiers and Russian sign names while keeping the surrounding interface in English.\')\nb,a=M[\'baseline\'],M[\'dinov2_augmented\']\np(f"On the held-out appearance-group test split, baseline accuracy was {b[\'accuracy\']:.3f} and macro-F1 was {b[\'macro_f1\']:.3f}. The augmented DINOv2 system achieved accuracy {a[\'accuracy\']:.3f} and macro-F1 {a[\'macro_f1\']:.3f}. These measurements describe this curated experiment, not performance on Kazakhstan roads.")\np(\'The application accepts a cropped sign upload or camera snapshot, returns class scores and a sign label, and displays nearest training examples. A conditions panel supports controlled darkening and blur. All reported measurements are generated from the experiment artifacts.\')\nfigure(\'examples.png\')\npage()\nheading(\'Dataset and preparation\')\np(\'The source is TN02/Road-Signs-Dataset [1]. Its authors describe a mixed collection built from GTSRB, RTSD, and 230 photographs captured in Baku, Azerbaijan. Consequently, the whole collection must not be described as locally photographed in Baku or Kazakhstan. The repository publishes images, Pascal VOC XML boxes, and an MIT license; upstream dataset attribution is retained.\')\ntable([[\'Sign identifier\',\'Class\',\'Train\',\'Validation\',\'Test\']]+[[c[\'code\'],c[\'en\'],M[\'counts\'][\'train\'][i],M[\'counts\'][\'val\'][i],M[\'counts\'][\'test\'][i]] for i,c in enumerate(json.loads((OUT/\'classes.json\').read_text()))],[65,150,70,90,65])\np(\'Images were read through their XML annotations. Selected boxes were clamped to image bounds; crops smaller than 16 pixels on either side were discarded. Each remaining sign was resized to 128 by 128 pixels for storage and to 224 by 224 for DINOv2. The five classes were selected because their symbols contain no numerical speed values or settlement names.\')\nfigure(\'dataset_balance.png\')\np(\'The paper-mini-sign and self-collected-data component of Assignment 1 was not completed. This project uses public data and reconstructs a baseline in the same experiment; it does not claim to continue previously trained personal weights.\',\'SmallText\')\npage()\nheading(\'Splitting and augmentation\')\np(\'Before splitting, each class was sampled to at most 1,200 candidate crops. Source filename variants and crops with a difference-hash Hamming distance of at most three were joined into appearance groups. A group was assigned wholly to train, validation, or test, with approximate proportions of 70, 15, and 15 percent. Each group contributes at most three crops. Per-class caps are 300 training, 80 validation, and 100 test crops. Seed 42 controls the sampling.\')\np(\'This reduces obvious duplication leakage, but cannot guarantee separation by physical sign or drive sequence: reliable provenance and track identifiers are unavailable in this mixed collection. Appearance grouping may also merge distinct signs with similar symbols. This is an explicit limitation of the evaluation, and the test set is not an independent Kazakhstan field benchmark.\')\np(\'The augmentation ablation adds one transformed training version per original crop. Transformations include modest perspective, rotation up to 12 degrees, brightness and contrast changes, and Gaussian blur. No mirrored sign labels or numerical symbols are synthesized. Validation and clean test images are not augmented during fitting.\')\np(\'Separate stress tests apply brightness factor 0.4 or Gaussian blur radius 2 to the held-out test crops. Their results measure sensitivity to these specific controlled corruptions. They do not represent actual rain, nighttime driving, or video capture.\')\nheading(\'Data flow\')\ntable([[\'Stage\',\'Output\'],[\'Public images and XML\',\'Validated sign crops and provenance manifest\'],[\'Appearance grouping\',\'Disjoint train, validation, and test groups\'],[\'Frozen encoder or HOG\',\'Numerical visual features\'],[\'Train-only fitting\',\'Saved classifier weights\'],[\'Held-out evaluation\',\'Metrics, confusion matrix, and stress tests\'],[\'Interactive application\',\'Prediction and nearest-example explanation\']],[170,270])\npage()\nheading(\'Modern vision component\')\np(\'DINOv2 [2] learns reusable visual representations through self supervision. During published pretraining, a student network learns to match a teacher across different image views; the teacher is updated by an exponential moving average. Global image and masked patch objectives encourage complementary information. This pretraining is supplied by Meta; it is not repeated in this project.\')\np(\'The ViT-S/14 encoder processes a 224 by 224 RGB crop as a 16 by 16 grid of patches and produces a 384-dimensional image embedding. The encoder is frozen, and a five-output linear head is trained on labeled signs. This separates expensive representation learning from inexpensive adaptation to the selected task.\')\nmethod_diagram()\np(\'The method is appropriate for a small labeled dataset because it reuses learned features instead of fitting a transformer from scratch. It also supports cosine-similarity retrieval for visual explanation. It is a modern foundation-model baseline, not a claim of best current traffic-sign accuracy. The measured comparison is decisive: clean accuracy matches HOG, while augmentation improves the dark-condition macro-F1 from 0.9936 to 0.9975. No broader superiority is claimed.\')\ntable([[\'Component\',\'Configuration\'],[\'Encoder\',\'DINOv2 ViT-S/14, frozen pretrained weights\'],[\'Input\',\'224 by 224 RGB; ImageNet normalization\'],[\'Feature scaling\',\'Training-feature mean and standard deviation only\'],[\'Classifier\',\'Linear head with five outputs\'],[\'Optimization\',\'AdamW; learning rate 0.008; weight decay 0.01\'],[\'Training\',\'45 full-batch epochs; lowest validation loss checkpoint\'],[\'Baseline\',\'HOG on 64 by 64 grayscale; standardized Logistic Regression, C 0.1\']],[145,295])\np(\'The baseline and both transformer heads share the same held-out test crops. The plain DINOv2 head uses original training crops; the augmented head uses original crops plus one transformed copy. Training curves concern the linear head only, not transformer pretraining.\')\npage()\nheading(\'Experiments and results\')\ntable([[\'Model\',\'Accuracy\',\'Macro F1\']]+[[label,f"{M[key][\'accuracy\']:.4f}",f"{M[key][\'macro_f1\']:.4f}"] for key,label in [(\'baseline\',\'HOG baseline\'),(\'dinov2_plain\',\'DINOv2 without augmentation\'),(\'dinov2_augmented\',\'DINOv2 with augmentation\')]],[260,90,90])\np(\'Accuracy counts the proportion of correctly classified crops. Macro-F1 gives every sign class equal weight, which helps expose weaker classes in an imbalanced selection. The classification report JSON contains per-class precision, recall, F1, and support. No detection mAP is reported because this version classifies a supplied crop instead of predicting boxes in full scenes.\')\nfigure(\'training_curves.png\',350)\nfigure(\'confusion_matrix.png\',260)\npage()\nheading(\'Robustness and explainability\')\ntable([[\'Condition\',\'Plain macro F1\',\'Augmented macro F1\']]+[[cond,f"{v[\'plain\'][\'macro_f1\']:.4f}",f"{v[\'augmented\'][\'macro_f1\']:.4f}"] for cond,v in M[\'conditions\'].items()],[180,130,130])\np(\'The ablation must be interpreted using the measured values: augmentation is not assumed to improve every condition. Different corruptions or an independent real-world evaluation may produce different conclusions.\')\nfigure(\'explanation.png\',455)\np(\'For visual explanation, cosine similarity retrieves the three closest original training embeddings to the input. These examples show what the representation finds similar; they are not a causal saliency map and do not prove the model used the correct visual evidence. Retrieval is restricted to training data.\')\np(f"Mean warmed single-image inference latency was {M[\'latency_ms_mean\']:.1f} ms across {M[\'latency_samples\']} test crops. Device: {M[\'hardware\'][\'gpu\'] or M[\'hardware\'][\'device\']}. Timing includes image preprocessing, feature extraction, and linear classification, with GPU synchronization; it excludes network transfer, browser rendering, and nearest-neighbor retrieval. Cold model download time is excluded.")\np(\'The app supports one cropped sign at a time. Camera input is a snapshot, not continuous live detection. Softmax is evaluated in float64 and scores are rendered as decimal strings at full stored precision rather than rounded integer percentages. This is a display change, not confidence calibration. Unsupported signs and unrelated objects can still receive a supported label.\')\npage()\nheading(\'Reproducibility and limitations\')\np(\'For the measured experiments, run the training notebook in Google Colab with a T4 GPU. For recovery without retraining, open Recovery_Colab.ipynb, upload the complete project ZIP, install dependencies, and launch the saved model. The bundle contains weights and the matching DINOv2 architecture code. The app can also run on CPU after dependencies are installed. The Gradio share link requires an active runtime.\')\np(\'The fixed seed improves repeatability but does not guarantee bit-identical GPU outputs. Package and hardware versions are recorded in environment files. Upstream repository updates and unavailable downloads may change reproducibility; the downloaded archive checksum is recorded by the notebook.\')\nheading(\'Ethics and intended use\')\np(\'This is an educational demonstration. It must not guide a moving vehicle or replace driver judgment. Public images can contain people, vehicle plates, or location cues; the app examples and training artifacts use cropped signs. No new identifiable photographs were collected. Synthetic brightness, blur, and geometry changes are disclosed as augmentations. Public hosting of the app should not be used for sensitive uploads.\')\np(\'The main limitations are mixed geographic provenance, incomplete capture metadata, possible residual duplicates, appearance-group rather than drive-level splitting, a small closed label set, no Kazakhstan field test, and no real paper-sign evaluation. These limitations prevent claims of deployment readiness or complete compliance with the self-collected-data requirement.\')\nheading(\'References\')\nfor s in [\n \'[1] Talibzade, Aliyev, Rafiyev and Khalilova. Road Signs Dataset. GitHub, 2023. https://github.com/TN02/Road-Signs-Dataset\',\n \'[2] Oquab et al. DINOv2 Learning Robust Visual Features without Supervision. 2023. https://arxiv.org/abs/2304.07193\',\n \'[3] Meta AI. Official DINOv2 implementation and pretrained models. https://github.com/facebookresearch/dinov2\',\n \'[4] Republic of Kazakhstan. Road Traffic Rules, Order 534 of 30 June 2023, Appendix 1. https://old.adilet.zan.kz/rus/docs/V2300033003\']:\n    p(s,\'SmallText\')\nSimpleDocTemplate(str(OUT/\'Report.pdf\'),pagesize=(595.28,841.89),leftMargin=50,rightMargin=50,topMargin=45,bottomMargin=48).build(story,onFirstPage=foot,onLaterPages=foot)\nprint(\'REPORT_CREATED\',flush=True)\n\n'
(OUT/'app.py').write_text(APP_SOURCE,encoding='utf-8')
(OUT/'build_report.py').write_text(REPORT_SOURCE,encoding='utf-8')
exec(compile(REPORT_SOURCE,'build_report.py','exec'),{'__file__':str(OUT/'build_report.py')})
# The real application video is recorded separately, not generated from plots.


## 4 Launch the application
Upload or capture a single cropped sign. The share link expires when the runtime stops. Do not upload sensitive images.

In [ ]:
import runpy
restored=runpy.run_path(str(OUT/'app.py'),run_name='saved_model_app')
demo=restored['demo']
demo.launch(share=True,debug=False,prevent_thread_lock=True,js=restored['ENGLISH_JS'],css='footer {display:none !important}')
restored['force_english'](demo)


## 5 Download results
Download the report, demonstration, and complete project archive.

In [ ]:
import shutil
from google.colab import files
archive=shutil.make_archive('/content/Traffic_Sign_Project','zip',str(OUT))
files.download(archive)
